# Regime-Sensitive Diebold-Li Estimation with Kalman Filtering

## A paper-style research notebook

**Purpose.** This notebook develops a regime-sensitive extension of the Diebold-Li term-structure model. It is written as a professional research document in English, with explicit LaTeX equations, methodological interpretation, and a fully executable empirical workflow.

The notebook is designed to answer a natural question that arises after estimating a standard dynamic Nelson-Siegel / Diebold-Li model:

> What happens if the latent factor dynamics are not stable over time, but instead change across distinct regimes?

To address that question, the notebook proceeds in two layers:

1. estimate the baseline Diebold-Li model with the Kalman filter;
2. augment that framework with regime classification and regime-specific factor dynamics.

The implementation below is intentionally transparent. It does **not** estimate a full hidden Markov state-space model under joint maximum likelihood. Instead, it constructs a serious intermediate specification that is empirically useful and easy to interpret.

## Abstract

This notebook extends the Diebold-Li yield curve model by allowing the dynamics of the latent factors to vary across regimes. The empirical strategy first estimates latent level, slope, and curvature factors under a standard linear Gaussian state-space system using the Kalman filter and Rauch-Tung-Striebel smoothing. The smoothed factors are then used to identify regime clusters through a Gaussian mixture model. Conditional on the estimated regime path, separate vector autoregressive dynamics are estimated for each regime. This framework makes it possible to compare spot and forward curve forecasts under alternative state evolutions and to quantify how persistence and cross-factor interactions differ across regimes. The resulting notebook provides a reproducible and research-oriented bridge between a standard dynamic Diebold-Li implementation and a more ambitious regime-switching term-structure model.

## 1. Motivation

Standard Diebold-Li models assume that the latent factor vector follows a single time-invariant transition law. In many applications, however, that assumption is too restrictive. Monetary policy cycles, financial stress episodes, inflation stabilization phases, and liquidity disruptions may all affect not only the level of the curve, but also the way in which level, slope, and curvature interact through time.

If such episodes exist, then a single transition matrix may average together dynamics that are in fact heterogeneous. This can lead to three problems:

1. factor persistence may be misestimated;
2. forecast paths may be biased toward an artificial "average regime";
3. economically meaningful changes in curve behavior may be hidden inside a single reduced-form law of motion.

A regime-sensitive framework is therefore appealing because it allows the latent-state dynamics to differ across episodes while keeping the cross-sectional yield-curve representation unchanged.

## 2. Baseline Diebold-Li system

We begin from the standard Diebold-Li representation:

$$
y_t(\tau) = \beta_{1,t} + \beta_{2,t}\left(\frac{1-e^{-\lambda \tau}}{\lambda \tau}\right) + \beta_{3,t}\left(\frac{1-e^{-\lambda \tau}}{\lambda \tau} - e^{-\lambda \tau}\right) + \varepsilon_t(\tau)
$$

or, in matrix form,

$$
y_t = \Lambda(\lambda) X_t + \varepsilon_t,
$$

where

$$
X_t = \begin{bmatrix} \beta_{1,t} \\ \beta_{2,t} \\ \beta_{3,t} \end{bmatrix}
$$

collects the level, slope, and curvature factors.

## 3. From a single dynamic law to regime-dependent dynamics

In the baseline model, factor dynamics are written as:

$$
X_t = c + A X_{t-1} + \eta_t, \qquad \eta_t \sim N(0,Q).
$$

A regime-sensitive extension instead allows the transition law to depend on a regime variable $S_t$:

$$
X_t = c_{S_t} + A_{S_t} X_{t-1} + \eta_{t,S_t}, \qquad \eta_{t,S_t} \sim N(0,Q_{S_t}).
$$

In a full hidden Markov setup, the regime process itself is latent and governed by a transition matrix. In this notebook, we use a simpler but still informative strategy:

1. estimate smoothed latent factors from the baseline Kalman filter;
2. classify those smoothed states into two regimes via Gaussian mixture clustering;
3. estimate separate transition dynamics for each regime;
4. use the empirical regime transition frequencies to generate regime-aware forecasts.

This approach does not claim to be a full Markov-switching state-space estimator. Rather, it provides a practical and interpretable approximation.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.linalg import solve_discrete_lyapunov
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler

plt.style.use('default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.grid'] = True

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent.parent

DATA_PATH = ROOT / 'api_js' / 'data' / 'market_rates.csv'

RATE_MONTHS = {
    'TPM': 1,
    'SPC_03Y': 3,
    'SPC_06Y': 6,
    'SPC_1Y': 12,
    'SPC_2Y': 24,
    'SPC_3Y': 36,
    'SPC_4Y': 48,
    'SPC_5Y': 60,
    'SPC_10Y': 120,
}

COLUMNS = list(RATE_MONTHS.keys())
TAU_MONTHS = np.array([RATE_MONTHS[c] for c in COLUMNS], dtype=float)
TAU_YEARS = TAU_MONTHS / 12.0

print(DATA_PATH)
print(COLUMNS)

In [ ]:
def load_monthly_panel(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, parse_dates=['Date'])
    monthly = (
        df.set_index('Date')[COLUMNS]
        .resample('ME')
        .last()
        .dropna(how='all')
    )
    complete = monthly.dropna(subset=COLUMNS).copy()
    complete.index.name = 'Date'
    return complete


def ns_loadings(tau_years: np.ndarray, lambda_value: float) -> np.ndarray:
    slope = (1.0 - np.exp(-lambda_value * tau_years)) / (lambda_value * tau_years)
    curvature = slope - np.exp(-lambda_value * tau_years)
    return np.column_stack([np.ones_like(tau_years), slope, curvature])


def ols_betas_for_panel(yields_df: pd.DataFrame, lambda_value: float):
    H = ns_loadings(TAU_YEARS, lambda_value)
    HtH_inv = np.linalg.inv(H.T @ H)
    betas = []
    residuals = []
    rmse = []
    for _, row in yields_df.iterrows():
        y = row[COLUMNS].to_numpy(dtype=float)
        beta = HtH_inv @ H.T @ y
        fitted = H @ beta
        err = y - fitted
        betas.append(beta)
        residuals.append(err)
        rmse.append(np.sqrt(np.mean(err ** 2)))
    betas = pd.DataFrame(betas, index=yields_df.index, columns=['level', 'slope', 'curvature'])
    residuals = pd.DataFrame(residuals, index=yields_df.index, columns=COLUMNS)
    return betas, residuals, float(np.mean(rmse))


def fit_var1(factors: pd.DataFrame):
    Y = factors.iloc[1:].to_numpy(dtype=float)
    Xlag = factors.iloc[:-1].to_numpy(dtype=float)
    X = np.column_stack([np.ones(len(Xlag)), Xlag])
    B = np.linalg.inv(X.T @ X) @ X.T @ Y
    c = B[0]
    A = B[1:].T
    resid = Y - X @ B
    Q = np.cov(resid.T, bias=False)
    return c, A, Q, resid


def kalman_filter_smoother(y_df, lambda_value, c, A, Q, R, x0=None, P0=None):
    H = ns_loadings(TAU_YEARS, lambda_value)
    Y = y_df[COLUMNS].to_numpy(dtype=float)
    nobs = Y.shape[0]
    nstate = A.shape[0]
    if x0 is None:
        x0 = np.linalg.solve(np.eye(nstate) - A, c)
    if P0 is None:
        P0 = solve_discrete_lyapunov(A, Q)

    x_pred = np.zeros((nobs, nstate))
    P_pred = np.zeros((nobs, nstate, nstate))
    x_filt = np.zeros((nobs, nstate))
    P_filt = np.zeros((nobs, nstate, nstate))

    x_prev = x0.copy()
    P_prev = P0.copy()
    for t in range(nobs):
        x_prior = c + A @ x_prev
        P_prior = A @ P_prev @ A.T + Q
        innovation = Y[t] - H @ x_prior
        S = H @ P_prior @ H.T + R
        K = P_prior @ H.T @ np.linalg.inv(S)
        x_post = x_prior + K @ innovation
        P_post = (np.eye(nstate) - K @ H) @ P_prior
        x_pred[t] = x_prior
        P_pred[t] = P_prior
        x_filt[t] = x_post
        P_filt[t] = P_post
        x_prev = x_post
        P_prev = P_post

    x_smooth = x_filt.copy()
    for t in range(nobs - 2, -1, -1):
        J = P_filt[t] @ A.T @ np.linalg.inv(P_pred[t + 1])
        x_smooth[t] = x_filt[t] + J @ (x_smooth[t + 1] - x_pred[t + 1])

    return pd.DataFrame(x_filt, index=y_df.index, columns=['level', 'slope', 'curvature']), pd.DataFrame(x_smooth, index=y_df.index, columns=['level', 'slope', 'curvature'])


def reconstruct_curve(months, beta_row, lambda_value):
    H = ns_loadings(np.asarray(months, dtype=float) / 12.0, lambda_value)
    return H @ np.asarray(beta_row, dtype=float)


def forward_curve_from_spot(months, spot_rates):
    months = np.asarray(months, dtype=float)
    spot = np.asarray(spot_rates, dtype=float) / 100.0
    forwards = []
    f_months = []
    for i in range(1, len(months)):
        t1 = months[i - 1] / 12.0
        t2 = months[i] / 12.0
        z1 = spot[i - 1]
        z2 = spot[i]
        f = ((1.0 + z2) ** t2 / (1.0 + z1) ** t1) ** (1.0 / (t2 - t1)) - 1.0
        forwards.append(f * 100.0)
        f_months.append(months[i])
    return np.asarray(f_months), np.asarray(forwards)

## 4. Baseline estimation

We begin with the same baseline estimation logic as in the standard paper-style notebook:

1. aggregate the panel to monthly frequency;
2. calibrate the decay parameter $\lambda$ by grid search;
3. estimate preliminary OLS factors;
4. estimate a VAR(1) on those factors;
5. recover filtered and smoothed states through the Kalman filter.

In [ ]:
panel = load_monthly_panel(DATA_PATH)
grid = np.linspace(0.02, 0.20, 120)
scores = []
for lam in grid:
    _, _, avg_rmse = ols_betas_for_panel(panel, lam)
    scores.append((lam, avg_rmse))

lambda_df = pd.DataFrame(scores, columns=['lambda', 'avg_rmse'])
best_lambda = float(lambda_df.sort_values('avg_rmse').iloc[0]['lambda'])
ols_betas, cross_residuals, avg_rmse = ols_betas_for_panel(panel, best_lambda)
state_intercept, transition_matrix, state_cov, _ = fit_var1(ols_betas)
measurement_cov = np.diag(np.maximum(cross_residuals.var(axis=0, ddof=1).to_numpy(dtype=float), 1e-6))

filtered_betas, smoothed_betas = kalman_filter_smoother(panel, best_lambda, state_intercept, transition_matrix, state_cov, measurement_cov)

best_lambda, avg_rmse, panel.shape

## 5. Regime identification through Gaussian mixtures

A full hidden-regime model would infer the regime and the latent factors jointly. That is beyond the scope of the current notebook. Instead, we use the smoothed latent factors as a low-noise representation of the term-structure state and identify regimes by clustering them.

Formally, we assume that the smoothed factor vector belongs to a finite mixture distribution:

$$
p(X_t) = \sum_{j=1}^{2} \pi_j \; \phi(X_t ; \mu_j, \Sigma_j),
$$

where:

- $\pi_j$ is the weight of regime $j$,
- $\mu_j$ is the regime-specific mean,
- $\Sigma_j$ is the regime-specific covariance matrix,
- $\phi(\cdot)$ is the multivariate Gaussian density.

We enrich the regime-identification feature set with a short-horizon volatility metric on the level factor, so that the clustering can capture not only location in factor space but also changes in local instability.

In [ ]:
regime_features = smoothed_betas.copy()
regime_features['d_level'] = regime_features['level'].diff()
regime_features['level_vol_6m'] = regime_features['d_level'].rolling(6).std()
regime_features = regime_features.drop(columns=['d_level']).dropna().copy()

scaler = StandardScaler()
X_regime = scaler.fit_transform(regime_features[['level', 'slope', 'curvature', 'level_vol_6m']])

gmm = GaussianMixture(n_components=2, covariance_type='full', random_state=42)
raw_labels = gmm.fit_predict(X_regime)
regime_probs = gmm.predict_proba(X_regime)

regime_frame = regime_features[['level', 'slope', 'curvature']].copy()
regime_frame['raw_regime'] = raw_labels

regime_means = regime_frame.groupby('raw_regime')['level'].mean().sort_values()
mapping = {old: new for new, old in enumerate(regime_means.index.tolist())}
regime_frame['regime'] = regime_frame['raw_regime'].map(mapping)

proba_cols = []
for raw in range(2):
    proba_cols.append((mapping[raw], regime_probs[:, raw]))
for final_regime, prob_series in sorted(proba_cols, key=lambda x: x[0]):
    regime_frame[f'prob_regime_{final_regime}'] = prob_series

regime_frame[['regime']].groupby('regime').size()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
axes[0].plot(smoothed_betas.index, smoothed_betas['level'], label='Smoothed level', color='tab:blue')
mask0 = regime_frame['regime'] == 0
mask1 = regime_frame['regime'] == 1
axes[0].scatter(regime_frame.index[mask0], regime_frame.loc[mask0, 'level'], s=20, label='Regime 0', alpha=0.75)
axes[0].scatter(regime_frame.index[mask1], regime_frame.loc[mask1, 'level'], s=20, label='Regime 1', alpha=0.75)
axes[0].set_title('Smoothed level factor and regime classification')
axes[0].legend(loc='best')

axes[1].plot(regime_frame.index, regime_frame['prob_regime_0'], label='P(Regime 0)')
axes[1].plot(regime_frame.index, regime_frame['prob_regime_1'], label='P(Regime 1)')
axes[1].set_title('Estimated regime probabilities')
axes[1].legend(loc='best')
plt.tight_layout()

## 6. Regime-specific transition dynamics

Once each observation has been assigned to a regime, we estimate a regime-specific transition law. For each regime $j$, we fit:

$$
X_t = c_j + A_j X_{t-1} + \eta_{t,j}, \qquad \eta_{t,j} \sim N(0,Q_j).
$$

This allows the persistence of level, slope, and curvature to vary across regimes, as well as the strength of cross-factor interactions.

We also compute empirical regime-transition frequencies, which serve as an approximation to the regime transition matrix:

$$
P = [p_{ij}], \qquad p_{ij} = \Pr(S_t = j \mid S_{t-1} = i).
$$

In this notebook, these probabilities are estimated by sample transition counts.

In [ ]:
regime_states = smoothed_betas.loc[regime_frame.index].copy()
regime_states['regime'] = regime_frame['regime']

def fit_var1_by_regime(states_df: pd.DataFrame, regime_value: int):
    sub = states_df.copy()
    Y = []
    Xlag = []
    for t in range(1, len(sub)):
        if sub['regime'].iloc[t] == regime_value and sub['regime'].iloc[t - 1] == regime_value:
            Y.append(sub[['level', 'slope', 'curvature']].iloc[t].to_numpy(dtype=float))
            Xlag.append(sub[['level', 'slope', 'curvature']].iloc[t - 1].to_numpy(dtype=float))
    Y = np.asarray(Y)
    Xlag = np.asarray(Xlag)
    if len(Y) < 10:
        raise ValueError(f'Not enough transitions for regime {regime_value}')
    X = np.column_stack([np.ones(len(Xlag)), Xlag])
    B = np.linalg.inv(X.T @ X) @ X.T @ Y
    c = B[0]
    A = B[1:].T
    resid = Y - X @ B
    Q = np.cov(resid.T, bias=False)
    return c, A, Q, resid, len(Y)


regime_models = {}
for r in [0, 1]:
    regime_models[r] = fit_var1_by_regime(regime_states, r)

transition_counts = np.zeros((2, 2), dtype=float)
labels = regime_frame['regime'].to_numpy(dtype=int)
for i in range(1, len(labels)):
    transition_counts[labels[i - 1], labels[i]] += 1
transition_probs = transition_counts / transition_counts.sum(axis=1, keepdims=True)

for r in [0, 1]:
    c_r, A_r, Q_r, _, nobs_r = regime_models[r]
    print('regime', r, 'usable transitions', nobs_r)
    print('intercept', np.round(c_r, 4))
    print('A')
    print(np.round(A_r, 4))
    print()

pd.DataFrame(transition_probs, index=['from_0', 'from_1'], columns=['to_0', 'to_1'])

## 7. Regime-aware forecasting

Two alternative forecasting strategies are considered.

### 7.1 Fixed-regime forecast

We assume that the current regime persists over the forecast horizon. This yields:

$$
E[X_{t+h}|\mathcal F_t, S_{t+k}=j\;\forall k\le h] = c_j + A_j E[X_{t+h-1}|\mathcal F_t, S_{t+k}=j\;\forall k\le h-1].
$$

### 7.2 Mixture-transition forecast

We instead allow the regime distribution to evolve according to the empirical transition matrix. The next state forecast becomes a probability-weighted average of regime-specific forecasts.

This second strategy is not a full hidden Markov forecast, but it is a useful approximation that incorporates the possibility of regime drift.

In [ ]:
def forecast_states_regime_fixed(last_state, c, A, steps):
    states = []
    x = np.asarray(last_state, dtype=float).copy()
    for _ in range(steps):
        x = c + A @ x
        states.append(x.copy())
    return np.asarray(states)


def forecast_states_mixture(last_state, regime_probs_now, regime_models, transition_probs, steps):
    x = np.asarray(last_state, dtype=float).copy()
    p = np.asarray(regime_probs_now, dtype=float).copy()
    states = []
    probs = []
    for _ in range(steps):
        next_p = p @ transition_probs
        forecasts = []
        for r in [0, 1]:
            c_r, A_r, _, _, _ = regime_models[r]
            forecasts.append(c_r + A_r @ x)
        x = next_p[0] * forecasts[0] + next_p[1] * forecasts[1]
        states.append(x.copy())
        probs.append(next_p.copy())
        p = next_p
    return np.asarray(states), np.asarray(probs)


last_date = regime_states.index[-1]
last_state = regime_states.loc[last_date, ['level', 'slope', 'curvature']].to_numpy(dtype=float)
last_regime = int(regime_states.loc[last_date, 'regime'])
last_probs = regime_frame.loc[last_date, ['prob_regime_0', 'prob_regime_1']].to_numpy(dtype=float)
horizons = [1, 3, 6, 12]
max_h = max(horizons)

fixed_forecast_states = forecast_states_regime_fixed(last_state, regime_models[last_regime][0], regime_models[last_regime][1], max_h)
mixture_forecast_states, future_regime_probs = forecast_states_mixture(last_state, last_probs, regime_models, transition_probs, max_h)

last_obs = panel.loc[last_date, COLUMNS].to_numpy(dtype=float)
current_curve = reconstruct_curve(TAU_MONTHS, last_state, best_lambda)
fixed_curves = {h: reconstruct_curve(TAU_MONTHS, fixed_forecast_states[h - 1], best_lambda) for h in horizons}
mix_curves = {h: reconstruct_curve(TAU_MONTHS, mixture_forecast_states[h - 1], best_lambda) for h in horizons}

last_date, last_regime, last_probs

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].plot(TAU_MONTHS, current_curve, linewidth=2.5, label=f'Current {last_date.date()}')
axes[0].scatter(TAU_MONTHS, last_obs, s=45, label='Observed')
for h in horizons:
    axes[0].plot(TAU_MONTHS, fixed_curves[h], marker='o', label=f'Fixed regime {h}M')
axes[0].set_title(f'Spot forecasts under fixed current regime ({last_regime})')
axes[0].set_xlabel('Maturity (months)')
axes[0].set_ylabel('Rate')
axes[0].legend(ncol=2)

axes[1].plot(TAU_MONTHS, current_curve, linewidth=2.5, label=f'Current {last_date.date()}')
axes[1].scatter(TAU_MONTHS, last_obs, s=45, label='Observed')
for h in horizons:
    axes[1].plot(TAU_MONTHS, mix_curves[h], marker='o', label=f'Mixture {h}M')
axes[1].set_title('Spot forecasts under regime-transition mixture')
axes[1].set_xlabel('Maturity (months)')
axes[1].set_ylabel('Rate')
axes[1].legend(ncol=2)

plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fm, ff = forward_curve_from_spot(TAU_MONTHS, current_curve)
axes[0].plot(fm, ff, linewidth=2.5, label='Current forward')
for h in horizons:
    fm_h, ff_h = forward_curve_from_spot(TAU_MONTHS, fixed_curves[h])
    axes[0].plot(fm_h, ff_h, marker='o', label=f'Fixed regime {h}M')
axes[0].set_title('Forward curves under fixed-regime forecast')
axes[0].set_xlabel('Forward maturity (months)')
axes[0].set_ylabel('Forward rate')
axes[0].legend(ncol=2)

axes[1].plot(fm, ff, linewidth=2.5, label='Current forward')
for h in horizons:
    fm_h, ff_h = forward_curve_from_spot(TAU_MONTHS, mix_curves[h])
    axes[1].plot(fm_h, ff_h, marker='o', label=f'Mixture {h}M')
axes[1].set_title('Forward curves under transition-mixture forecast')
axes[1].set_xlabel('Forward maturity (months)')
axes[1].set_ylabel('Forward rate')
axes[1].legend(ncol=2)

plt.tight_layout()

In [ ]:
future_prob_df = pd.DataFrame(
    future_regime_probs[[h - 1 for h in horizons]],
    index=[f'{h}M' for h in horizons],
    columns=['P(regime 0)', 'P(regime 1)'],
)
future_prob_df

## 8. Interpretation

The regime-sensitive framework should be read as a structured diagnostic tool.

### If the two regimes look genuinely different

then the data suggest that the latent term-structure dynamics are not stable across the sample. In that case, the regime-aware forecasts may contain meaningful information that would be lost under a single-transition model.

### If the two regimes look very similar

then the clustering step may simply be partitioning noise, and the baseline Diebold-Li state-space system may already be sufficient.

### Economically useful patterns to watch for

- a high-level regime with stronger persistence in the level factor;
- a steepening regime in which slope shocks are amplified rather than damped;
- a higher-volatility regime in which curvature becomes unstable;
- a divergence between fixed-regime and transition-mixture forecasts.

These patterns help determine whether the curve should be interpreted under a single stable law of motion or a state-dependent one.

## 9. Limitations

This notebook remains deliberately transparent and therefore deliberately simplified.

It does **not**:

- estimate a hidden Markov model jointly with the Kalman state-space system;
- maximize a single likelihood over regime transitions and latent states at the same time;
- allow the measurement equation itself to switch by regime;
- construct full predictive density intervals.

Instead, it should be understood as an empirical bridge between:

- a standard dynamic Diebold-Li model, and
- a full regime-switching term-structure model.

That makes it especially useful for exploratory research, model diagnostics, and the generation of economically interpretable scenarios.

## 10. Conclusion

A standard Diebold-Li model imposes a single dynamic law on the latent factor vector. That is often a useful first approximation, but not always a satisfactory one. The regime-sensitive framework in this notebook shows how one can move beyond that restriction while preserving the analytical clarity of the original model.

By combining Kalman smoothing, Gaussian mixture classification, regime-specific VAR dynamics, and regime-aware spot and forward forecasts, the notebook produces a coherent extension of the baseline yield-curve framework. It is not yet the final word in regime-switching term-structure estimation, but it is already a strong and reproducible research platform for investigating whether the dynamics of the curve are genuinely state dependent.